# Telegram Channel Migration — complete developer handoff

**Updated:** 13 November 2026  
**Repository:** https://github.com/kawakaze430/telegram-migretion  
**Deliverable:** This is the only project notebook. It contains the frozen migration plan, caption cleaner, photo manifest, Telegram uploader, usage instructions, manual recovery, and developer context. Developers do **not** need the original ChatGPT conversations to understand the approved workflow.

## 1. Project goal and user-approved decisions

Migrate the **entire available historical Telegram movie-post collection** into the intended destination channel using Google Colab. Preserve all historical photo posts, including older versions and duplicates; **never collapse them to one movie/title**. Produce clean destination posts with:

- **Original image/photo** (unchanged); **movie/series name**, **genre**, and **audio/language description** whenever present in the original.
- **EVERY available legitimate non-channel file/download URL**, in its original sequence. Keep 480p, 720p, 1080p, HEVC, 4K, size-labelled files, alternate hosts and multiple mirrors. **Never require a 1080p link.** Posts that stop at 480p stop there; posts that stop at 720p stop there. A resolution label alone is not a download URL.
- Remove **all Telegram channel/invite/post URLs**, `tg://` links, hidden clickable channel links, `@username` promotions, WhatsApp channel links, and channel-promotion text throughout the caption (before, between, or after download URLs).
- Keep source order and content history. Do **not** invent missing names, genre, audio, or resolutions. A photo without a download link is still a photo to preserve.
- Clean caption text only. Do not modify image pixels, watermarks/text printed inside photos, or external hosts.
- Do not edit or delete any existing messages in the destination; this is a new, separately tracked upload run.

**Example of intended behavior** (illustrative placeholder URLs, not source data):

```text
Sample Movie (2026)
📼Genre: #Action | #Drama
🔊Audio: Hindi
480p: https://example.org/file-480
720p: https://example.org/file-720
```

If only 480p was present, preserve only that link. If 1080p mirrors or unlabelled links exist, preserve them too. Any source channel-promotion lines are removed.

## 2. Frozen source and exact scope

**Frozen source:** [Movie1 GitHub export at commit 3f8ff02](https://github.com/kawakaze430/Movie1/tree/3f8ff02f9b0a417205c317c36d2eaa73356955bf). The notebook embeds the cleaned export and a photo manifest for this precise commit. This is deliberate: source changes must not silently alter a resumable run.

| Item | Verified preparation count |
| --- | ---: |
| Available source posts | **3,792** |
| Outgoing posts | **3,780** |
| Photos to preserve | **3,751** |
| Remaining text-only posts | **29** |
| Text-only promotion posts made empty and skipped | **12** |
| Promotion photos preserved with empty captions | **12** |
| Photos lacking an original non-channel download link | **84** |
| Channel-link occurrences removed | **9,132** |
| `@username` occurrences removed | **61** |
| Non-channel URL occurrences kept | **9,463** |
| Older/duplicate photo posts removed | **0** |
| Missing exported video files | **29** |
| Non-post/service records omitted | **2** |
| Original photo bytes to fetch | **409,476,705** (about 410 MB) |

These are *snapshot-specific checks*, not a guarantee that the source will never change, that off-site links work, or that Telegram accepts every upload.

**Why 3,780 rather than 3,792?** Twelve text-only promotion messages become empty after mandatory channel-link removal. The 3,780-plan includes every one of the 3,751 photo posts. The 29 absent video files are separately reported as a source/export limitation, not fabricated or uploaded.

## 3. Architecture (all inside this notebook)

1. **Prepare, offline/no sends.** Mount Drive when on Colab; otherwise use a local working folder. Verify the embedded ZIP checksum, unpack the frozen `result.all_downloads.json`, `all_photos.json`, `caption_review.json`, `skipped_records.json`, `summary.json`, and Python modules `clean_captions.py` and `uploader.py`. Do not overwrite an unexpectedly changed prepared file.
2. **Validate.** `uploader.validate_plan` and `cleaner.remaining_channel_references` check outgoing records before network sends. The cleaner preserves source text entities and recalculates UTF-16 entity offsets where necessary; source URL occurrences are reviewed in `caption_review.json`.
3. **Review.** Preview selected post IDs and all exceptions in step 2. Read `caption_review.json`, `skipped_records.json`, and `summary.json` in the Drive working directory when debugging.
4. **Authenticate and select target.** A Telegram bot, added as administrator of the destination channel with posting permission, uses the official Bot API. Enter its token in the hidden prompt or the Colab secret `TELEGRAM_BOT_TOKEN`. Never paste secrets into the notebook or commit them to GitHub. Step 3 only verifies access.
5. **Send five posts** using the exact displayed `SEND 5 TO <destination title>` confirmation. Manually inspect both photo and caption in the destination channel.
6. **Upload the remainder** using the exact `UPLOAD ALL TO <destination title>` confirmation. Uploads are sequential, throttled (normally ≥3 seconds apart), checkpointed, and resumable. The first five are not sent again. Telegram network/rate-limit timing can vary.
7. **Recovery, if interrupted.** Progress is saved under `Telegram_Movie1_All_Links_No_Channels_v2` on Drive. An uncertain in-flight delivery remains **pending** and is **never automatically retried**: inspect Telegram, then use the recovery cell to mark it as sent with the real message ID or not sent. Remove a stale lock only after stopping all upload processes. Back up the prepared data/progress if desired; backups contain no bot token.

### Detailed behavioral acceptance criteria

- Source order and all **3,751** original photo references are retained; no post is dropped because it lacks 720p/1080p, has no download URL, or resembles an older duplicate.
- Every source movie title, genre, audio, and all **9,463** non-channel URLs remain. URLs are not rewritten or guessed. If metadata follows a link, preserve it.
- Zero outgoing Telegram/WhatsApp channel links or promotional `@username` references, including embedded link entities. Do not truncate legitimate links following 1080p or after a specific number of lines.
- An empty text-only promotion is skipped; a photo whose caption becomes empty is **still uploaded as a photo**.
- Do not upload the 29 unavailable videos unless those missing original files are independently recovered and a *new versioned plan* is tested.
- No network sends during preparation, preview or connection checks; only explicit send cells can post. Never send to the old source channel.
- Each completed post has durable progress, but an ambiguous/failed send must not be silently replayed. Preserve the progress folder between Colab restarts.
- A new snapshot or modified cleaning logic needs a new versioned plan, new tests, and distinct progress storage; do not overwrite existing in-progress output.

## 4. How to run this Colab notebook

1. Open this `.ipynb` via GitHub → **Open in Colab**, or upload it to Google Colab. Choose **CPU / None**; a GPU is unnecessary.
2. Run **1. Prepare** (loads the frozen plan) and **2. Preview**. Check exception records and confirm the resulting formatting/links.
3. Add a bot as a channel administrator, run **3. Connect**, and verify the printed **destination channel ID and title**. Ensure the target is *your* destination, not the source or the wrong channel.
4. Run **4. Send five** only after reviewing and typing the exact confirmation. Check the actual posts in Telegram.
5. Run **5. Upload remaining** only after the five-post test succeeds. Keep one uploader active and Colab running; if interrupted, rerun preparation and connection, resolve pending deliveries, then resume.

**Security/permissions:** The repository is **public**. This notebook embeds prepared captions, off-site URLs, source IDs, and a photo manifest (although no Telegram bot token). Only distribute media and file links that you have the right to repost. The tool will not verify copyright ownership, availability of external downloads, or the contents of images.

## 5. Decisions that were explicitly superseded

- **Old approach:** a 2,573-post pass plus a separate 1,219-post duplicate/older-image pass (3,792 candidates). **Not current.** All qualifying photo posts now live in one plan; no duplicate filter.
- **Old approach:** stop text after 1080p. **Cancelled.** Preserve all available quality links, sizes and mirrors, stopping at the last retained link except for any metadata after it.
- **Old approach:** retain promotion/channel links. **Cancelled.** Remove them completely.
- Do **not** run the earlier `Telegram_Movie1_Uploader.ipynb` or `Telegram_Movie1_Extra_1219.ipynb` alongside this one. They have different selection and progress semantics.

## 6. What is and is not tested

The prior preparation reported 20 local tests covering resolution variants/mirrors, hidden link entities, channel cleanup at arbitrary positions, metadata preservation, promotion-only posts, message offset handling, rate limits, interrupted-send recovery, resumption and notebook preparation using a mocked Telegram API. It **did not** perform an authenticated end-to-end live Telegram/Colab migration. The five-post verification is an essential real-world test; do not claim 3,780 live posts have been migrated just because the preparation completed.

**Developer change policy:** Implement future fixes in this one notebook; update the snapshot and acceptance counts deliberately, keep previous progress untouched, and document what changed in a new top section. Do not add unrelated scripts/files to this repository unless the owner changes the single-notebook requirement.

---

## Executable notebook

The cells below are the reviewed **12 November 2026, all-links/no-channels** migration workflow. Follow them in order.


# Movie1 — keep ALL download links, remove ALL channel links

**Revised 12 November 2026. This replaces the 1080p-cutoff version.**

Keep every original photo, movie name, genre, audio description, and non-channel
download link: **480p, 720p, 1080p, HEVC, 4K, size-labelled files, and every mirror**.
A movie with only a 480p or 720p link is kept. There is no quality-based cutoff.
Movie captions end at their last retained URL, with any later genre/audio information
also preserved. Earlier channel links are removed too.

Removed throughout all captions: Telegram invites/public links/post links, hidden
clickable Telegram links, `tg://` links, `@username` mentions, WhatsApp channel links,
and the accompanying channel-promotion lines. No original photo is edited.

### What will upload

- **3,780 posts: all 3,751 photos + 29 remaining text posts.** No duplicates or older
  photo posts are filtered out.
- **9,132 channel-link occurrences and 61 username mentions removed.** All **9,463
  non-channel URL occurrences** are preserved exactly, including download mirrors.
- 12 text-only promotion posts became empty and are skipped. Their originals remain
  in the review/skip files. Another 12 promotion photos are retained without captions.
- 84 original photo posts had no non-channel download URL. Their photos and remaining
  text are kept; missing movie names/genre/audio/downloads are not invented. Review
  these exceptions in step 2.
- 29 video files were absent from the export; those remain unavailable. Two service
  events are not uploaded.

**Stop all older upload notebooks first.** This version uses a new progress folder
and starts a full new upload. It does not edit/delete earlier Telegram posts or reuse
old upload progress. Use an empty destination if you want only this cleaned version.

Open in Google Colab with a **CPU / None runtime**, then run steps 1–5. Photos download
automatically from the inspected GitHub snapshot. Your token is entered privately.
No message is sent until you type a destination-specific send confirmation.


In [ ]:
#@title 1. Prepare all posts with every download link (no Telegram sends)
from pathlib import Path
import base64, hashlib, importlib.util, io, json, sys, zipfile
try:
    from google.colab import drive
except ImportError:
    WORKDIR = Path.cwd() / 'Telegram_Movie1_All_Links_No_Channels_v2'
else:
    drive.mount('/content/drive')
    WORKDIR = Path('/content/drive/MyDrive/Telegram_Movie1_All_Links_No_Channels_v2')
WORKDIR.mkdir(parents=True, exist_ok=True)
BUNDLE_B64 = '...'
EXPECTED_BUNDLE_SHA256 = '63e7a987d64457a8aefb484d514994dbefb0df06f6a2470666b4857077173f02'

package_bytes = base64.b64decode(BUNDLE_B64)
if hashlib.sha256(package_bytes).hexdigest() != EXPECTED_BUNDLE_SHA256:
    raise ValueError('Package checksum failed.')
with zipfile.ZipFile(io.BytesIO(package_bytes)) as package:
    for item in package.infolist():
        if Path(item.filename).name != item.filename:
            raise ValueError('Unexpected package path.')
        data = package.read(item)
        destination = WORKDIR / item.filename
        if destination.exists() and destination.read_bytes() != data:
            raise ValueError('Prepared file differs: ' + item.filename + '. Keep progress; do not overwrite it.')
        if not destination.exists():
            destination.write_bytes(data)
def load_module(name, filename):
    spec = importlib.util.spec_from_file_location(name, WORKDIR / filename)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module
uploader = load_module('movie1_no_channels_uploader', 'uploader.py')
cleaner = load_module('movie1_channel_cleaner', 'clean_captions.py')
MESSAGES = json.loads((WORKDIR / 'result.all_downloads.json').read_text(encoding='utf-8'))['messages']
MANIFEST = json.loads((WORKDIR / 'all_photos.json').read_text(encoding='utf-8'))
REVIEW = json.loads((WORKDIR / 'caption_review.json').read_text(encoding='utf-8'))
SUMMARY = json.loads((WORKDIR / 'summary.json').read_text(encoding='utf-8'))
uploader.validate_plan(MESSAGES, MANIFEST)
if len(MESSAGES) != SUMMARY['upload_posts'] or any(cleaner.remaining_channel_references(m) for m in MESSAGES):
    raise ValueError('Prepared caption validation failed. No posts sent.')
print('Ready:', len(MESSAGES), 'posts:', SUMMARY['photo_posts'], 'photos +', SUMMARY['text_posts'], 'text posts.')
print('All duplicates, older photo posts, and non-channel download links are retained.')
print('Removed channel-link occurrences:', SUMMARY['channel_link_occurrences_removed'])
print('Preserved non-channel URL occurrences:', SUMMARY['non_channel_link_occurrences_preserved'])
print('Text-only promotions now empty and skipped:', SUMMARY['empty_text_posts_skipped'])
print('Photo posts that originally lack a non-channel download link:', SUMMARY['photo_posts_without_non_channel_link'])
print('Progress folder:', WORKDIR)
print('Photo downloads: approximately', round(SUMMARY['photo_bytes'] / 1_000_000), 'MB. No messages sent.')


## 2. Preview the cleaned captions

The examples include movies ending at 480p, 720p, size-labelled links, and multiple
1080p mirrors. Enter any source ID to inspect that caption. All non-channel links
remain; links to Telegram channels are removed even if labelled as movie downloads.
Missing information is left missing. This cell makes no Telegram requests.


In [ ]:
#@title 2. Preview outgoing captions and exceptions (no uploads)
SOURCE_ID_TO_PREVIEW = 0 #@param {type:"integer"}
SHOW_ALL_MISSING_DOWNLOAD_TITLES = False #@param {type:"boolean"}
preview_ids = [SOURCE_ID_TO_PREVIEW] if SOURCE_ID_TO_PREVIEW else [8, 17, 32, 57, 3138]
for mid in preview_ids:
    rows = [r for r in REVIEW if r['source_id'] == mid]
    if not rows:
        print('Source ID not found:', mid)
        continue
    row = rows[0]
    print('\nSOURCE', mid, '| links kept:', len(row['non_channel_links_preserved']))
    print('Channel links removed:', row['channel_links_removed'], '| usernames removed:', row['mentions_removed'])
    print(row['upload_text'] or ('(Empty text-only promotion: skipped)' if row['empty_text_post'] else '(Photo retained without a caption)'))
exceptions = [r for r in REVIEW if r['photo_without_non_channel_link']]
print('\nPHOTOS THAT HAD NO EXTERNAL DOWNLOAD LINK IN THE ORIGINAL:', len(exceptions))
for row in exceptions if SHOW_ALL_MISSING_DOWNLOAD_TITLES else exceptions[:12]:
    print(row['source_id'], '|', row['title'][:130])
print('\nFull before/after review:', WORKDIR / 'caption_review.json')
print('This version keeps all 3,751 original photos. No messages sent.')


## 3. Connect your bot and destination channel

Add the bot as an administrator with permission to post. Your previous channel ID is
filled in. Change it if you are using an empty new channel. A public `@username` also
works. Clearing the field lists recent channels known to the bot; if none appear,
post `MIGRATION_SETUP` in the destination after adding the bot as admin, then rerun.

Enter the bot token in the hidden prompt, or use the Colab secret
`TELEGRAM_BOT_TOKEN`. This step only verifies access; it sends no channel messages.


In [ ]:
#@title 3. Connect (no uploads)
CHANNEL_ID_OR_USERNAME = "-1004492918137" #@param {type:"string"}
from getpass import getpass
token = None
try:
    from google.colab import userdata
    token = userdata.get('TELEGRAM_BOT_TOKEN')
except Exception:
    pass
if not token:
    token = getpass('Bot token (hidden): ')
API = uploader.TelegramAPI(token)
del token
target = str(CHANNEL_ID_OR_USERNAME).strip()
if not target:
    channels = uploader.discover_channels(API)
    for channel in channels:
        print(channel['id'], channel.get('title', ''))
    if not channels:
        raise ValueError('No recent channel update. Add the bot as admin, post MIGRATION_SETUP in the destination, then rerun step 3.')
    target = input('Enter the numeric ID of your intended destination channel: ').strip()
if target.lstrip('-').isdigit():
    target = int(target)
BOT, CHANNEL = uploader.validate_target(API, target)
if CHANNEL['id'] == -1001881963160:
    raise ValueError('This is the old source channel. Choose your intended destination.')
STATE_PATH = WORKDIR / ('progress_' + str(CHANNEL['id']) + '.json')
CACHE = WORKDIR / 'photo_cache'
state = uploader.load_state(STATE_PATH, MESSAGES, BOT['id'], CHANNEL['id'])
print('Destination:', CHANNEL['title'], '| ID:', CHANNEL['id'])
print('Bot:', BOT.get('username'), '| Completed in THIS version:', len(state['sent']), '/ 3780')
if state.get('pending'):
    print('Pending delivery needs the recovery cell:', state['pending'])
print('This plan uploads 3,780 posts, including ALL 3,751 photos. Existing channel posts are not edited or deleted.')
print('Stop all older upload notebooks before continuing. Connection verified; no posts sent.')


## 4. Send five posts and check them in Telegram

The next five unsent records are part of the full plan and will not be repeated by
step 5. Inspect their photos, movie details, download links, and removed promotions.
Stop here if the destination or result is wrong. No deletion is performed.


In [ ]:
#@title 4. Send the next five posts
expected = 'SEND 5 TO ' + CHANNEL['title']
answer = input('To send the next five posts, type exactly: ' + expected + '\n')
if answer == expected:
    state = uploader.run_upload(API, MESSAGES, MANIFEST, STATE_PATH, CACHE,
                                BOT['id'], CHANNEL['id'], limit=5, interval=3.0)
else:
    print('No posts sent.')


## 5. Continue all remaining posts / resume this version

Progress is saved after each successful send in the new Drive folder. Resume with
the same bot, channel, and folder. Never delete progress or run two uploaders at once.
The programmed pauses total about **3 hours 9 minutes**; transfers and Telegram waits
add time. Keep Colab open and your computer awake/online. Runtime availability is not
guaranteed; the ChatGPT tab can be closed.

After an interruption, rerun steps 1 and 3, resolve any pending send using the recovery
cell, then rerun step 5. Uncertain deliveries are never automatically resent.


In [ ]:
#@title 5. Upload every remaining post / resume this version
expected = 'UPLOAD ALL TO ' + CHANNEL['title']
answer = input('To upload the remaining posts, type exactly: ' + expected + '\n')
if answer == expected:
    state = uploader.run_upload(API, MESSAGES, MANIFEST, STATE_PATH, CACHE,
                                BOT['id'], CHANNEL['id'], limit=0, interval=3.0)
else:
    print('No posts sent.')


## Recovery — only after an interrupted upload

Rerun steps 1 and 3 first. Stop all upload processes before clearing a stale lock.
If a send is pending, inspect Telegram before deciding whether it arrived. This cell
only records your finding and sends no messages. Keep the progress folder intact.


In [ ]:
#@title Recover this version's pending send or stale lock (no messages sent)
lock = Path(str(STATE_PATH) + '.lock')
if lock.exists():
    answer = input('After stopping ALL upload processes, type ALL UPLOADS STOPPED to clear a stale lock: ')
    if answer != 'ALL UPLOADS STOPPED':
        raise ValueError('Lock retained; no progress changed.')
    lock.unlink()
if STATE_PATH.exists():
    state = uploader.load_state(STATE_PATH, MESSAGES, BOT['id'], CHANNEL['id'])
    pending = state.get('pending')
    if pending:
        print('Inspect this post in Telegram:', pending)
        action = input('After checking Telegram, type sent or not_sent: ').strip()
        new_id = None
        if action == 'sent':
            new_id = int(input('Actual NEW message ID (last number in its copied message link): ').strip())
        if action not in ('sent', 'not_sent'):
            raise ValueError('No delivery status changed.')
        uploader.resolve_pending(STATE_PATH, pending['source_id'], action, new_id)
        print('Recovery saved. Continue with step 5.')
    else:
        print('No pending send. Continue with step 5.')
else:
    print('This version has not started uploading yet.')


## Optional: back up cleaned captions, original review, and progress

Run only while no upload is active. This backup contains prepared data and progress,
including the before/after review and skipped originals. It excludes the bot token
and cached photos. Keep the Drive progress folder for resuming.


In [ ]:
#@title Download prepared data and progress backup (no Telegram sends)
if list(WORKDIR.glob('progress_*.json.lock')):
    raise ValueError('Stop the upload before making a progress backup.')
backup_path = WORKDIR / 'Telegram_Movie1_All_Links_No_Channels_Backup.zip'
names = ['result.all_downloads.json', 'all_photos.json', 'caption_review.json',
         'skipped_records.json', 'summary.json', 'clean_captions.py', 'uploader.py']
with zipfile.ZipFile(backup_path, 'w', zipfile.ZIP_DEFLATED) as package:
    for name in names:
        package.write(WORKDIR / name, arcname=name)
    for path in WORKDIR.glob('progress_*.json'):
        package.write(path, arcname=path.name)
try:
    from google.colab import files
    files.download(str(backup_path))
except ImportError:
    print('Backup:', backup_path)


## What was verified

Every available source record was checked. All 3,751 photo references, all source
movie-title/genre/audio lines in movie posts, and every non-channel URL were preserved.
No duplicate photo post was filtered out. Visible captions and hidden link entities
were scanned for remaining channel URLs and username mentions; none remain in the
outgoing plan. UTF-16 formatting positions are recalculated after cleanup.

Local tests cover multiple qualities/mirrors, captions without resolution labels,
Telegram links at any position, hidden links, WhatsApp channels, metadata preservation,
empty captions, rate limits, interrupted-send recovery, progress resumption, and actual
notebook preparation with a mocked Telegram API. No authenticated live Telegram or
end-to-end Colab upload was performed during preparation. Use the five-post step to
check that integration. External download availability was not tested. Photos are
unchanged; text printed inside an image is outside this caption cleanup.

[Source snapshot](https://github.com/kawakaze430/Movie1/tree/3f8ff02f9b0a417205c317c36d2eaa73356955bf)
· [Telegram photo API](https://core.telegram.org/bots/api#sendphoto)
· [Colab limitations](https://research.google.com/colaboratory/faq.html)
